Jupyter Notebook for the Kaggle Competition (Machine Learning - CentraleSupélec - January 2025)  
PAES DE ALMEIDA NINA DUARTE, Pedro ; COLLIER, Grégoire ; PERARDT MAGALHÃES BRITO, Romero

Import necessary packages

In [174]:
import geopandas as gpd
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np


from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score

Import data (as shown in the baseline code) and treat NaN

In [175]:
change_type_map = {'Demolition': 0, 'Road': 1, 'Residential': 2, 'Commercial': 3, 'Industrial': 4,
                   'Mega Projects': 5}

## Read csvs

train_df = gpd.read_file('train.geojson', index_col=0)
test_df = gpd.read_file('test.geojson', index_col=0)

train_y = train_df['change_type'].apply(lambda x: change_type_map[x])

c:\Users\pedro\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver GeoJSON does not support open option INDEX_COL
  return ogr_read(
c:\Users\pedro\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver GeoJSON does not support open option INDEX_COL
  return ogr_read(


In [176]:
features = train_df.columns.to_list()
features = [feature for feature in features if feature != 'change_type']

In [177]:
train_df.shape

(296146, 45)

Fill NaN with mean

In [178]:
ignore_columns = ['urban_type', 'geography_type', 'change_type','date0', 'change_status_date0', 'date1',
                   'change_status_date1','date2', 'change_status_date2','date3', 'change_status_date3', 
                   'date4', 'change_status_date4', 'index', 'geometry']

for col in train_df.columns:
    if col not in ignore_columns:
        train_df[col].fillna(train_df[col].mean(), inplace=True)

C:\Users\pedro\AppData\Local\Temp\ipykernel_18684\2416107534.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df[col].fillna(train_df[col].mean(), inplace=True)


Fill missing dates with closest date (one above)

In [179]:
train_df['date0'].fillna(method='ffill', inplace=True)
train_df['date1'].fillna(method='ffill', inplace=True)
train_df['date2'].fillna(method='ffill', inplace=True)
train_df['date3'].fillna(method='ffill', inplace=True)
train_df['date4'].fillna(method='ffill', inplace=True)

C:\Users\pedro\AppData\Local\Temp\ipykernel_18684\4175239564.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['date0'].fillna(method='ffill', inplace=True)
C:\Users\pedro\AppData\Local\Temp\ipykernel_18684\4175239564.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  train_df['date0'].fillna(method='ffill', inplace=True)
C:\Users\pedro\AppData\Local\Temp\ipykernel_18684\4175239564.py:2: FutureWarning: A value is trying to be s

In [180]:
train_df.isna().any()

urban_type              False
geography_type          False
change_type             False
img_red_mean_date1      False
img_green_mean_date1    False
img_blue_mean_date1     False
img_red_std_date1       False
img_green_std_date1     False
img_blue_std_date1      False
img_red_mean_date2      False
img_green_mean_date2    False
img_blue_mean_date2     False
img_red_std_date2       False
img_green_std_date2     False
img_blue_std_date2      False
img_red_mean_date3      False
img_green_mean_date3    False
img_blue_mean_date3     False
img_red_std_date3       False
img_green_std_date3     False
img_blue_std_date3      False
img_red_mean_date4      False
img_green_mean_date4    False
img_blue_mean_date4     False
img_red_std_date4       False
img_green_std_date4     False
img_blue_std_date4      False
img_red_mean_date5      False
img_green_mean_date5    False
img_blue_mean_date5     False
img_red_std_date5       False
img_green_std_date5     False
img_blue_std_date5      False
date0     

Drop 1458 rows that had NaN in change_status columns

In [181]:
#train_df.dropna(inplace=True)
train_df.fillna('Greenland')

,urban_type,geography_type,change_type,img_red_mean_date1,img_green_mean_date1,img_blue_mean_date1,img_red_std_date1,img_green_std_date1,img_blue_std_date1,img_red_mean_date2,...,date1,change_status_date1,date2,change_status_date2,date3,change_status_date3,date4,change_status_date4,index,geometry
0,Sparse Urban,"Dense Forest,Grass Land",Road,93.371775,107.291113,89.827379,29.812040,28.328368,25.324294,125.773062,...,09-12-2013,Greenland,10-09-2016,Construction Started,22-07-2019,Construction Done,24-07-2017,Construction Midway,0,"POLYGON ((112.16774 32.02198, 112.16845 32.020..."
1,Sparse Urban,"Dense Forest,Grass Land",Road,96.071674,107.061702,90.755556,24.896240,22.275180,22.080686,133.097679,...,09-12-2013,Greenland,10-09-2016,Land Cleared,22-07-2019,Construction Done,24-07-2017,Construction Midway,1,"POLYGON ((112.16849 32.02048, 112.16891 32.019..."
2,Sparse Urban,"Dense Forest,Grass Land",Road,101.212148,113.462178,95.670574,24.179684,21.873401,21.285197,120.713490,...,09-12-2013,Greenland,10-09-2016,Land Cleared,22-07-2019,Construction Done,24-07-2017,Land Cleared,2,"POLYGON ((112.16892 32.01969, 112.16962 32.018..."
3,Rural,"Dense Forest,Grass Land",Road,94.463311,99.995531,84.470046,26.869852,23.767679,19.351983,114.819776,...,09-12-2013,Greenland,10-09-2016,Construction Started,22-07-2019,Construction Done,24-07-2017,Construction Midway,3,"POLYGON ((112.16966 32.0181, 112.17033 32.0166..."
4,Dense Urban,"Sparse Forest,Dense Forest,Farms",Demolition,151.883646,191.710197,211.569244,52.465332,59.441844,52.304349,141.514462,...,09-12-2013,Prior Construction,10-09-2016,Prior Construction,22-07-2019,Land Cleared,24-07-2017,Prior Construction,4,"POLYGON ((112.16669 32.01597, 112.16677 32.015..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296141,"N,A",Coastal,Commercial,239.297084,229.193482,215.205832,25.969706,31.586712,32.155574,140.346141,...,25-02-2017,Construction Done,27-01-2014,Land Cleared,28-03-2018,Construction Done,28-12-2015,Land Cleared,296141,"POLYGON ((-109.72152 23.00665, -109.72155 23.0..."
296142,Sparse Urban,"Coastal,Sparse Forest",Residential,162.912319,143.865217,122.935145,56.127846,44.184674,49.760802,103.760507,...,25-02-2017,Construction Done,27-01-2014,Greenland,28-03-2018,Construction Done,28-12-2015,Land Cleared,296142,"POLYGON ((-109.7161 23.01983, -109.71617 23.01..."
296143,Sparse Urban,Dense Forest,Residential,111.304320,94.723404,80.374597,21.540545,17.786801,18.143091,68.845906,...,25-02-2017,Construction Done,27-01-2014,Greenland,28-03-2018,Construction Done,28-12-2015,Greenland,296143,"POLYGON ((-109.71829 23.02784, -109.71832 23.0..."
296144,Sparse Urban,Dense Forest,Residential,137.374613,136.108359,113.544892,32.344779,30.077877,29.759516,98.718266,...,25-02-2017,Land Cleared,27-01-2014,Land Cleared,28-03-2018,Construction Midway,28-12-2015,Land Cleared,296144,"POLYGON ((-109.71771 23.02794, -109.71768 23.0..."


Treat 'N,A' in 'urban_type' and 'geography_type' column

In [182]:
## Used codes for urban_type and geography_type
train_df[train_df['urban_type'] == 'N,A']
train_df[train_df['geography_type'].str.contains('Snow')]['urban_type'].value_counts()
train_df['urban_type'].value_counts() 

urban_type
Dense Urban                           89427
Sparse Urban                          69189
Industrial                            60411
N,A                                   36682
Rural                                 20428
Sparse Urban,Industrial                8129
Dense Urban,Industrial                 7223
Sparse Urban,Urban Slum                1948
Urban Slum                             1472
Dense Urban,Urban Slum                  669
Rural,Industrial                        188
Urban Slum,Industrial                   181
Sparse Urban,Dense Urban                 74
Dense Urban,Rural                        52
Sparse Urban,Urban Slum,Industrial       47
Sparse Urban,Rural                       20
Urban Slum,Rural                          6
Name: count, dtype: int64

In [183]:
train_df.loc[((train_df['urban_type'] == 'N,A') & (train_df['geography_type'].str.contains("Dense Forest|Farms"))), 'urban_type'] = 'Rural'
train_df.loc[((train_df['urban_type'] == 'N,A') & (train_df['geography_type'].str.contains("Barren Land"))), 'urban_type'] = 'Urban'
train_df.loc[((train_df['urban_type'] == 'N,A') & (train_df['geography_type'].str.contains("Sparse Forest"))), 'urban_type'] = 'Sparse Urban'


## Costal had 3545 N,A, 136 Industrial, 32 Sparse Urban and 19 Dense Urban
train_df.loc[((train_df['urban_type'] == 'N,A') & (train_df['geography_type'].str.contains("Coastal"))), 'urban_type'] = 'Industrial'

## River had 664 Industrial, 215 Dense Urban and 115 N,A
train_df.loc[((train_df['urban_type'] == 'N,A') & (train_df['geography_type'].str.contains("River"))), 'urban_type'] = 'Industrial'

## Desert had 1520 N,A, 1350 Industrial, 1257 Sparse Urban and 940 Dense Urban
train_df.loc[((train_df['urban_type'] == 'N,A') & (train_df['geography_type'].str.contains("Desert"))), 'urban_type'] = 'Industrial'

## Lakes or Green Land had 15669 Sparse Urban and 237 N,A
train_df.loc[((train_df['urban_type'] == 'N,A') & (train_df['geography_type'].str.contains("Lakes|Green Land"))), 'urban_type'] = 'Sparse Urban'

## Grass Land had 41861 Dense Urban, 33048 Sparse Urban and 271 N,A
train_df.loc[((train_df['urban_type'] == 'N,A') & (train_df['geography_type'].str.contains("Grass Land"))), 'urban_type'] = 'Dense Urban'

## Snow had 4 Sparse Urban, 3 Dense Urban and 3 N,A
train_df.loc[((train_df['urban_type'] == 'N,A') & (train_df['geography_type'].str.contains("Snow"))), 'urban_type'] = 'Sparse Urban'

In [184]:
## Unideal substitution
train_df.loc[train_df['geography_type'] == 'N,A', 'geography_type'] = 'Sparse Forest'

In [185]:
train_df_encoded = pd.get_dummies(train_df, 
                                  columns=['urban_type', 'geography_type', 'change_status_date0', 'change_status_date1', 'change_status_date2', 'change_status_date3', 'change_status_date4'],
                                   drop_first=False, dtype=float)
train_df_encoded

,change_type,img_red_mean_date1,img_green_mean_date1,img_blue_mean_date1,img_red_std_date1,img_green_std_date1,img_blue_std_date1,img_red_mean_date2,img_green_mean_date2,img_blue_mean_date2,...,change_status_date4_Construction Done,change_status_date4_Construction Midway,change_status_date4_Construction Started,change_status_date4_Excavation,change_status_date4_Greenland,change_status_date4_Land Cleared,change_status_date4_Materials Dumped,change_status_date4_Materials Introduced,change_status_date4_Operational,change_status_date4_Prior Construction
0,Road,93.371775,107.291113,89.827379,29.812040,28.328368,25.324294,125.773062,139.833243,134.900701,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Road,96.071674,107.061702,90.755556,24.896240,22.275180,22.080686,133.097679,145.385190,137.092518,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Road,101.212148,113.462178,95.670574,24.179684,21.873401,21.285197,120.713490,131.633447,124.436492,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,Road,94.463311,99.995531,84.470046,26.869852,23.767679,19.351983,114.819776,127.827828,120.435373,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Demolition,151.883646,191.710197,211.569244,52.465332,59.441844,52.304349,141.514462,171.079581,181.960612,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296141,Commercial,239.297084,229.193482,215.205832,25.969706,31.586712,32.155574,140.346141,116.700172,98.334477,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
296142,Residential,162.912319,143.865217,122.935145,56.127846,44.184674,49.760802,103.760507,81.104710,70.001087,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
296143,Residential,111.304320,94.723404,80.374597,21.540545,17.786801,18.143091,68.845906,62.948420,51.315925,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
296144,Residential,137.374613,136.108359,113.544892,32.344779,30.077877,29.759516,98.718266,85.318885,72.572755,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [186]:
train_df_encoded['geometry'] = train_df_encoded['geometry'].area
train_df_encoded['geometry']

C:\Users\pedro\AppData\Local\Temp\ipykernel_18684\921578829.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  train_df_encoded['geometry'] = train_df_encoded['geometry'].area
C:\Users\pedro\AppData\Local\Temp\ipykernel_18684\921578829.py:1: UserWarning: Geometry column does not contain geometry.
  train_df_encoded['geometry'] = train_df_encoded['geometry'].area


0         8.174601e-07
1         4.394334e-07
2         8.209702e-07
3         8.175168e-07
4         1.484970e-07
              ...     
296141    1.532601e-08
296142    1.943119e-08
296143    1.090439e-08
296144    1.806174e-09
296145    2.520225e-09
Name: geometry, Length: 296146, dtype: float64

In [189]:
preprocessed_df = train_df_encoded.drop(['change_type', 'index', 'date0', 'date1', 'date2', 'date3', 'date4'], axis=1)
train_y

0         1
1         1
2         1
3         1
4         0
         ..
296141    3
296142    2
296143    2
296144    2
296145    2
Name: change_type, Length: 296146, dtype: int64

In [194]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(preprocessed_df, train_y, train_size=0.3, random_state = 0)

from sklearn.tree import DecisionTreeClassifier
dtree_model = DecisionTreeClassifier(max_depth = 30).fit(X_train, y_train)
dtree_predictions = dtree_model.predict(X_test)

# creating a confusion matrix
cm = confusion_matrix(y_test, dtree_predictions)

cm

array([[15007,   308,  3963,  2724,    54,     3],
       [  316,  2203,  3986,  3561,    47,     7],
       [ 4658,  3187, 67769, 27942,   315,    33],
       [ 2961,  3103, 29337, 34247,   528,    30],
       [   48,    43,   309,   462,    45,     0],
       [    9,     6,    47,    43,     2,     0]], dtype=int64)

In [195]:
## Initial submission